# Amazon Business Entity Resolution — Cloud/GPU Pipeline

This notebook runs the complete **end-to-end pipeline** (Blocking, Feature Engineering, LightGBM Classifier with Macro $F_{0.5}$ Tuning, and Submission Packaging) on a free GPU (Google Colab / Kaggle).

### Recommended Runtime
- In Google Colab: Go to **Runtime -> Change runtime type -> T4 GPU** (free tier).

## 1. Clone Repository and Install Dependencies

In [ ]:
!git clone https://github.com/SANKETHEGADE/business-entity-resolution.git
%cd business-entity-resolution/business_entity_resolution

!pip install -q pandas numpy scikit-learn rapidfuzz unidecode tqdm lightgbm sentence-transformers

## 2. Place or Extract Dataset Files

Upload your downloaded dataset zip file into Colab and unzip it into `dataset/train` and `dataset/test`.
Or if files are in Google Drive, mount Drive:

In [ ]:
# If using Google Drive:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r /content/drive/MyDrive/amazon_data/* dataset/

# Or if you uploaded a zip directly to Colab:
# !unzip -q /content/dataset.zip -d dataset/

# Verify directory contents:
!ls -la dataset/train/
!ls -la dataset/test/

## 3. Train Model & Tune Macro $F_{0.5}$ Threshold

Runs dual-channel selective blocking, extracts 20+ features, and tunes the decision threshold on a held-out validation slice.

In [ ]:
!python -m src.pipeline --split train

## 4. Run Test Inference & Generate Submission Files

Generates both required files:
- `output/matching_results.tsv` (Leaderboard upload)
- `output/candidate_pairs.tsv` (Audit candidate set for ranking)

In [ ]:
!python -m src.pipeline --split test

## 5. Validate Output Files Against Challenge Rules

Runs `utils/validate_submission.py` to guarantee zero format errors or rejected predictions.

In [ ]:
!python utils/validate_submission.py \
    --matching output/matching_results.tsv \
    --candidate output/candidate_pairs.tsv \
    --test-dir dataset/test

## 6. Build `<team_name>_submission.zip` & Download Outputs

In [ ]:
# Replace 'my_team' with your team's name
!python utils/make_submission_zip.py --team-name my_team --test-dir dataset/test

# Download final files to your local machine
from google.colab import files
files.download('output/matching_results.tsv')
# files.download('../my_team_submission.zip')